# Governance Dashboard — QuickSight Analysis for Inference & Drift Monitoring

> 📊 **Best viewed on [nbviewer](https://nbviewer.org/github/aws-samples/sample-mlops-bestpractices/blob/main/sagemaker-automated-drift-and-trend-monitoring/notebooks/4_governance_dashboard.ipynb)** — GitHub's renderer strips JavaScript that powers interactive output cells (Evidently reports, plotly charts, ipywidgets). nbviewer (run by Project Jupyter) renders them in full.


This notebook programmatically creates a complete QuickSight dashboard — no manual UI steps.

**Schema awareness:** the feature-drift dataset (Sheet 3) and the inference dataset both expose `monitoring_run_id`. The drift Lambda back-fills this column on `inference_responses` for every row it scored, so the cross-dataset JOIN is now an exact foreign key (was a 24-hour time-window approximation in earlier versions).

**Visuals — Sheet 1 (Inference Monitoring):**
1. Prediction Volume Over Time
2. Fraud Probability Distribution
3. Prediction Accuracy Breakdown
4. Risk Tier Distribution
5. Inference Latency Trend
6. Total Inferences KPI
7. Model Prediction Accuracy Over Time (%) — NEW
8. Daily Predictions: Correct vs Incorrect — NEW

**Visuals — Sheet 2 (Drift Trend Analysis):**
7. Data Drift Share Over Time (line chart)
8. Drifted Columns Count Trend (bar chart)
9. Model Drift Detection (ROC-AUC comparison)
10. Data Drift Score Distribution (histogram)
11. Monitoring Run Volume (bar chart)
12. Drift Detection KPI (latest drift share)

**Visuals — Sheet 3 (Feature Drift Analysis by Model Version):**
13. Drift by Model Version Over Time (multi-line showing drift evolution per version)
14. Model Version Performance Summary (table with drift, ROC-AUC, run counts)
15. Inference Volume by Model Version (stacked area chart)
16. Drift Intensity Heatmap (pivot table: time × version)

**Visuals — Sheet 4 (Feature-Level Drift Detail):**
17. Top Drifted Features (bar chart)
18. Feature Drift Score Trend (line chart with filter)
19. Feature Drift Details (table)
20. Feature Stability Overview (KPI card)
21. Feature Drift Distribution (histogram)
22. Feature Drift Heatmap (pivot: features × time)


## What this notebook does

This notebook builds (or refreshes) the QuickSight **governance dashboard** — the
inference, drift, feature-drift, and prediction-accuracy visuals — on top of the
Athena tables written by the drift-monitoring pipeline.

**All of the logic lives in one place:** `src/governance/create_governance_dashboard.py`.
That module is the single source of truth for every dataset, Athena view, visual, analysis,
and dashboard definition. It is also what `main.py dashboard create` calls. This notebook is
a thin driver over that module — it does **not** redefine any datasets or visuals inline, so
the notebook and the CLI can never drift out of sync.

> Because everything is defined in the module, the feature-drift visuals automatically use the
> test-agnostic **`drift_magnitude`** field (higher = more drift, regardless of which statistical
> test Evidently picked) instead of the ambiguous raw `drift_score`.

## 1. Setup & Configuration

In [1]:
import sys, boto3
from pathlib import Path
from dotenv import load_dotenv

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
load_dotenv(project_root / '.env')

from src.config.config import AWS_DEFAULT_REGION, QUICKSIGHT_IDENTITY_REGION, ATHENA_DATABASE

# Everything the notebook does is delegated to this module — the single
# source of truth for datasets, the feature_drift_detail view, visuals,
# the analysis, and the dashboard. No dataset/visual is redefined here.
from src.governance import create_governance_dashboard as gov

# Account ID for the QuickSight/Athena calls below.
sts = boto3.client('sts', region_name=AWS_DEFAULT_REGION)
ACCOUNT_ID = sts.get_caller_identity()['Account']

print(f'Account:                    {ACCOUNT_ID}')
print(f'Data-plane region (Athena): {AWS_DEFAULT_REGION}')
print(f'QuickSight identity region: {QUICKSIGHT_IDENTITY_REGION}')
print(f'Database:                   {ATHENA_DATABASE}')

Account:                  329430715989
Data-plane region (Athena): us-east-1
QuickSight identity region: us-east-1
Database:                 fraud_detection


> ⚠️ **First-time QuickSight setup required (once per AWS account).**
>
> If this is a brand-new account, `describe_account_settings` in the next cell will fail with `ResourceNotFoundException` or similar. QuickSight has to be signed up for in the AWS console first — the CLI/notebook can't do it for you. **5-minute one-time setup:**
>
> 1. AWS console → search **QuickSight** → **Sign up for QuickSight**.
> 2. Edition: **Enterprise** (Standard doesn't support the Definition API this notebook uses).
> 3. Auth: **Use IAM federated identities and QuickSight-managed users** (default).
> 4. Region: pick your **identity region** — QuickSight fixes this per account at sign-up. Usually `us-east-1`. If yours is different, set `QUICKSIGHT_IDENTITY_REGION=<region>` in `.env` before running this notebook.
> 5. QuickSight console → username (top right) → **Manage QuickSight** → **Manage users** → **Invite users** — add your IAM/SSO identity as **Author** or **Admin**.
> 6. Same page → **Security & permissions** → **Manage QuickSight access to AWS services** → check **Amazon S3** (select the base stack's data bucket) and **Amazon Athena**. Save.
>
> Verify by visiting [https://quicksight.aws.amazon.com/](https://quicksight.aws.amazon.com/) — you should see the QuickSight home page. Then rerun the cell below.
>
> Full prerequisites + troubleshooting: see the README's **QuickSight prerequisites (one-time per account)** section.

## 2. Verify QuickSight Subscription

In [2]:
try:
    qs = quicksight_admin.describe_account_settings(AwsAccountId=ACCOUNT_ID)
    edition = qs['AccountSettings'].get('Edition', 'Unknown')
    print(f'\u2713 QuickSight active (Edition: {edition})')
    if edition == 'STANDARD':
        print('  \u26a0 Definition API requires Enterprise edition')
except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceNotFoundException':
        print('\u2717 QuickSight not subscribed: https://quicksight.aws.amazon.com/')
    else: raise

✓ QuickSight active (Edition: ENTERPRISE)


## 3. Verify Inference Data in Athena

In [3]:
## 2. Build the dashboard (one shot)

`gov.create_dashboard()` runs the entire flow end-to-end, in order:

print(f'\nChecking {ATHENA_MONITORING_RESPONSES_TABLE} table...')
try:
    r = run_athena_query(f'SELECT COUNT(*) FROM {ATHENA_MONITORING_RESPONSES_TABLE}')
    drift_count = r['ResultSet']['Rows'][1]['Data'][0]['VarCharValue']
    print(f'  Total monitoring runs: {drift_count}')
    r = run_athena_query(f'SELECT MIN(monitoring_timestamp), MAX(monitoring_timestamp) FROM {ATHENA_MONITORING_RESPONSES_TABLE}')
    print(f'  Range: {r["ResultSet"]["Rows"][1]["Data"][0].get("VarCharValue","N/A")} \u2192 {r["ResultSet"]["Rows"][1]["Data"][1].get("VarCharValue","N/A")}')
    print('\u2713 Drift data available')
    HAS_DRIFT_DATA = int(drift_count) > 0
except Exception as e:
    print(f'  \u26a0 monitoring_responses not available: {e}')
    print('  Drift trend visuals will be empty until monitoring runs complete.')
    HAS_DRIFT_DATA = False

Checking inference_responses table...


  Total records: 105


  With ground truth: 105


  Range: 2026-07-16 16:57:23.738562 → 2026-07-16 17:00:01.868952
✓ Data available

Checking monitoring_responses table...


  Total monitoring runs: 6


  Range: 2026-07-15 23:48:07.000000 → 2026-07-16 20:06:00.000000
✓ Drift data available


## 4. Create Athena Data Source in QuickSight

In [4]:
def get_quicksight_principals():
    """Return ARNs for ALL QuickSight users so any account user can access assets."""
    try:
        users = quicksight_admin.list_users(AwsAccountId=ACCOUNT_ID, Namespace='default')
        arns = [u['Arn'] for u in users.get('UserList', [])]
        if arns:
            return arns
    except Exception:
        pass
    return [f'arn:aws:quicksight:{QUICKSIGHT_IDENTITY_REGION}:{ACCOUNT_ID}:user/default/Admin/*']

QS_PRINCIPALS = get_quicksight_principals()
# Keep QS_PRINCIPAL for backward compat (first user) but grant to all
QS_PRINCIPAL = QS_PRINCIPALS[0]
print(f'QuickSight principals ({len(QS_PRINCIPALS)} users):')
for p in QS_PRINCIPALS:
    print(f'  {p}')

This is idempotent — re-run it any time to pick up code changes in the module.

If QuickSight shows **"Lake Formation permissions missing"** when opening the analysis,
uncomment and run the commands below. This grants `IAM_ALLOWED_PRINCIPALS` access so any
IAM role with Glue/Athena permissions (including QuickSight's service role) can query the tables.

QuickSight principals (1 users):
  arn:aws:quicksight:us-east-1:329430715989:user/default/Admin/kruthira-Isengard
Updating existing data source...


✓ Data source: arn:aws:quicksight:us-east-1:329430715989:datasource/fraud-governance-athena-datasource


In [5]:
# ============================================================================
# LAKE FORMATION & S3 PERMISSIONS FIX
# ============================================================================
# Run this cell if QuickSight shows "Lake Formation permissions missing"
# or "s3:GetObject permission denied" errors.
#
# To grant permissions, call the functions at the bottom of this cell.

import subprocess
import json
import boto3

def grant_lakeformation_permissions(database, tables=None, grant_database=True):
    """
    Grant Lake Formation permissions to IAM_ALLOWED_PRINCIPALS.
    
    Args:
        database: Database name (from ATHENA_DATABASE config)
        tables: List of table names to grant permissions on. If None, only grants database-level.
        grant_database: If True, grants ALL on database first
    
    Returns:
        dict: Results with success/failure status for each resource
    """
    results = {'database': None, 'tables': {}}
    
    def run_grant(resource_json, permissions, resource_name):
        """Execute lakeformation grant-permissions command"""
        cmd = [
            'aws', 'lakeformation', 'grant-permissions',
            '--principal', '{"DataLakePrincipalIdentifier":"IAM_ALLOWED_PRINCIPALS"}',
            '--resource', resource_json,
            '--permissions'
        ] + permissions + ['--region', REGION]
        
        try:
            result = subprocess.run(cmd, capture_output=True, text=True, check=True)
            perms_str = ', '.join(permissions)
            print(f"  ✓ {resource_name}")
            print(f"    Granted: {perms_str}")
            return True
        except subprocess.CalledProcessError as e:
            error_msg = e.stderr.strip()
            # Ignore "already exists" errors
            if 'AlreadyExistsException' in error_msg or 'already granted' in error_msg.lower():
                perms_str = ', '.join(permissions)
                print(f"  ℹ {resource_name}")
                print(f"    Already granted: {perms_str}")
                return True
            else:
                print(f"  ✗ {resource_name}")
                print(f"    Error: {error_msg}")
                return False
    
    print("="*80)
    print(f"LAKE FORMATION PERMISSIONS - Database: {database}")
    print("="*80)
    
    # 1. Grant database-level permissions
    if grant_database:
        resource = json.dumps({"Database": {"Name": database}})
        results['database'] = run_grant(resource, ["ALL"], f"Database: {database}")
        print()
    
    # 2. Grant table-level permissions
    if tables:
        table_permissions = ["SELECT", "DESCRIBE", "ALTER", "DROP", "INSERT", "DELETE"]
        print(f"Granting table permissions ({len(tables)} tables):")
        for i, table in enumerate(tables, 1):
            print(f"\n[{i}/{len(tables)}] Table: {database}.{table}")
            resource = json.dumps({"Table": {"DatabaseName": database, "Name": table}})
            results['tables'][table] = run_grant(
                resource, 
                table_permissions, 
                f"{table}"
            )
    
    return results

def grant_s3_permissions(role_name, bucket_name):
    """
    Grant S3 permissions to QuickSight service role using boto3.
    
    Args:
        role_name: IAM role name (from QUICKSIGHT_SERVICE_ROLE_NAME config)
        bucket_name: S3 bucket name (from DATA_S3_BUCKET config)
    
    Returns:
        dict: Results with success/failure for each policy
    """
    iam = boto3.client('iam', region_name=REGION)
    results = {}
    
    print("\n" + "="*80)
    print(f"S3 PERMISSIONS - Role: {role_name}")
    print("="*80)
    
    # 1. Data lake access policy
    data_lake_policy = {
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Action": ["s3:GetObject", "s3:ListBucket", "s3:GetBucketLocation"],
            "Resource": [
                f"arn:aws:s3:::{bucket_name}",
                f"arn:aws:s3:::{bucket_name}/*"
            ]
        }]
    }
    
    try:
        iam.put_role_policy(
            RoleName=role_name,
            PolicyName='QuickSightS3DataLakeAccess',
            PolicyDocument=json.dumps(data_lake_policy)
        )
        print(f"  ✓ Policy: QuickSightS3DataLakeAccess")
        print(f"    Bucket: {bucket_name}")
        print(f"    Actions: s3:GetObject, s3:ListBucket, s3:GetBucketLocation")
        results['data_lake'] = True
    except Exception as e:
        print(f"  ✗ Policy: QuickSightS3DataLakeAccess")
        print(f"    Error: {e}")
        results['data_lake'] = False
    
    # 2. Athena query results access policy
    athena_results_policy = {
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Action": ["s3:GetObject", "s3:PutObject", "s3:ListBucket", "s3:GetBucketLocation"],
            "Resource": [
                "arn:aws:s3:::aws-athena-query-results-*",
                "arn:aws:s3:::aws-athena-query-results-*/*"
            ]
        }]
    }
    
    try:
        iam.put_role_policy(
            RoleName=role_name,
            PolicyName='QuickSightAthenaResultsAccess',
            PolicyDocument=json.dumps(athena_results_policy)
        )
        print(f"\n  ✓ Policy: QuickSightAthenaResultsAccess")
        print(f"    Bucket: aws-athena-query-results-* (all regions)")
        print(f"    Actions: s3:GetObject, s3:PutObject, s3:ListBucket, s3:GetBucketLocation")
        results['athena_results'] = True
    except Exception as e:
        print(f"\n  ✗ Policy: QuickSightAthenaResultsAccess")
        print(f"    Error: {e}")
        results['athena_results'] = False
    
    return results

# ============================================================================
# To grant permissions, uncomment and run the code below:
# ============================================================================
#
TABLES_TO_GRANT = [
    ATHENA_INFERENCE_TABLE,
    ATHENA_MONITORING_RESPONSES_TABLE,
    ATHENA_GROUND_TRUTH_UPDATES_TABLE,
]

lf_results = grant_lakeformation_permissions(
    database=ATHENA_DATABASE,
    tables=TABLES_TO_GRANT,
    grant_database=True
)

s3_results = grant_s3_permissions(
    role_name=QUICKSIGHT_SERVICE_ROLE_NAME,
    bucket_name=DATA_S3_BUCKET
)

print("\n" + "="*80)
print("✅ PERMISSIONS SETUP COMPLETE")
print("="*80)

print("Permissions functions defined. Uncomment code above to grant permissions.")

LAKE FORMATION PERMISSIONS - Database: fraud_detection


  ✗ Database: fraud_detection
    Error: aws: [ERROR]: An error occurred (AccessDeniedException) when calling the GrantPermissions operation: Resource does not exist or requester is not authorized to access requested permissions.

Granting table permissions (3 tables):

[1/3] Table: fraud_detection.inference_responses


  ✓ inference_responses
    Granted: SELECT, DESCRIBE, ALTER, DROP, INSERT, DELETE

[2/3] Table: fraud_detection.monitoring_responses


  ✓ monitoring_responses
    Granted: SELECT, DESCRIBE, ALTER, DROP, INSERT, DELETE

[3/3] Table: fraud_detection.ground_truth_updates


  ✓ ground_truth_updates
    Granted: SELECT, DESCRIBE, ALTER, DROP, INSERT, DELETE

S3 PERMISSIONS - Role: aws-quicksight-service-role-v0
  ✓ Policy: QuickSightS3DataLakeAccess
    Bucket: fraud-detection-monitoring-data-329430715989
    Actions: s3:GetObject, s3:ListBucket, s3:GetBucketLocation

  ✓ Policy: QuickSightAthenaResultsAccess
    Bucket: aws-athena-query-results-* (all regions)
    Actions: s3:GetObject, s3:PutObject, s3:ListBucket, s3:GetBucketLocation

✅ PERMISSIONS SETUP COMPLETE
Permissions functions defined. Uncomment code above to grant permissions.


## 6. Create Datasets

Two datasets: `inference_responses` for prediction data, `monitoring_responses` for drift metrics.
Uses `RelationalTable` so QuickSight auto-discovers columns from Athena.
Calculated fields (`prediction_accuracy`, `risk_tier`) are added via `LogicalTableMap`.

In [6]:
# RelationalTable — no column list needed, QuickSight auto-discovers from Athena
physical_table_map = {
    'inference-responses': {
        'RelationalTable': {
            'DataSourceArn': DATASOURCE_ARN,
            'Catalog': 'AwsDataCatalog',
            'Schema': ATHENA_DATABASE,
            'Name': ATHENA_INFERENCE_TABLE,
            'InputColumns': [
                {'Name': 'inference_id', 'Type': 'STRING'},
                {'Name': 'request_timestamp', 'Type': 'DATETIME'},
                {'Name': 'endpoint_name', 'Type': 'STRING'},
                {'Name': 'model_version', 'Type': 'STRING'},
                {'Name': 'mlflow_run_id', 'Type': 'STRING'},
                {'Name': 'input_features', 'Type': 'STRING'},
                {'Name': 'prediction', 'Type': 'INTEGER'},
                {'Name': 'probability_positive', 'Type': 'DECIMAL'},
                {'Name': 'probability_non_fraud', 'Type': 'DECIMAL'},
                {'Name': 'confidence_score', 'Type': 'DECIMAL'},
                {'Name': 'ground_truth', 'Type': 'INTEGER'},
                {'Name': 'ground_truth_timestamp', 'Type': 'DATETIME'},
                {'Name': 'ground_truth_source', 'Type': 'STRING'},
                {'Name': 'days_to_ground_truth', 'Type': 'DECIMAL'},
                {'Name': 'inference_latency_ms', 'Type': 'DECIMAL'},
                {'Name': 'model_load_time_ms', 'Type': 'DECIMAL'},
                {'Name': 'preprocessing_time_ms', 'Type': 'DECIMAL'},
                {'Name': 'transaction_id', 'Type': 'STRING'},
                {'Name': 'transaction_amount', 'Type': 'DECIMAL'},
                {'Name': 'customer_id', 'Type': 'STRING'},
                {'Name': 'is_high_confidence', 'Type': 'BIT'},
                {'Name': 'is_low_confidence', 'Type': 'BIT'},
                {'Name': 'prediction_bucket', 'Type': 'STRING'},
                {'Name': 'request_id', 'Type': 'STRING'},
                {'Name': 'response_time', 'Type': 'DATETIME'},
                {'Name': 'error_message', 'Type': 'STRING'},
                {'Name': 'inference_mode', 'Type': 'STRING'},
                # monitoring_run_id back-fills here when a drift run scores this row.
                # NULL until the drift Lambda's UPDATE statement tags it. Joins to
                # monitoring_responses.monitoring_run_id 1:N for "which inferences
                # contributed to this drift run?" lookups.
                {'Name': 'monitoring_run_id', 'Type': 'STRING'},
            ],
        }
    }
}

# Calculated fields via LogicalTableMap
logical_table_map = {
    'inference-responses-logical': {
        'Alias': 'Inference Responses',
        'Source': {'PhysicalTableId': 'inference-responses'},
        'DataTransforms': [
            {
                'CreateColumnsOperation': {
                    'Columns': [
                        {
                            'ColumnName': 'prediction_accuracy',
                            'ColumnId': 'prediction-accuracy',
                            'Expression': (
                                "ifelse("
                                "isNull({ground_truth}), 'Pending', "
                                "ifelse({prediction} = {ground_truth}, 'Correct', 'Incorrect'))"
                            ),
                        },
                        {
                            'ColumnName': 'risk_tier',
                            'ColumnId': 'risk-tier',
                            'Expression': (
                                "ifelse("
                                "{probability_positive} > 0.8, 'High Risk', "
                                "ifelse({probability_positive} > 0.5, 'Medium Risk', "
                                "ifelse({probability_positive} > 0.2, 'Low Risk', 'Minimal Risk')))"
                            ),
                        },
                    ]
                }
            }
        ],
    }
}

DSET_ACTIONS = [
    'quicksight:DescribeDataSet', 'quicksight:DescribeDataSetPermissions',
    'quicksight:PassDataSet', 'quicksight:DescribeIngestion',
    'quicksight:ListIngestions', 'quicksight:UpdateDataSet',
    'quicksight:DeleteDataSet', 'quicksight:CreateIngestion',
    'quicksight:CancelIngestion', 'quicksight:UpdateDataSetPermissions',
]
dset_common = dict(
    AwsAccountId=ACCOUNT_ID, DataSetId=DATASET_ID,
    Name=QUICKSIGHT_INFERENCE_DATASET_NAME,
    PhysicalTableMap=physical_table_map,
    LogicalTableMap=logical_table_map,
    ImportMode='DIRECT_QUERY',
)
try:
    quicksight.describe_data_set(AwsAccountId=ACCOUNT_ID, DataSetId=DATASET_ID)
    print('Updating existing dataset...')
    resp = quicksight.update_data_set(**dset_common)
except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceNotFoundException':
        print('Creating new dataset...')
        resp = quicksight.create_data_set(
            **dset_common,
            Permissions=[{'Principal': p, 'Actions': DSET_ACTIONS} for p in QS_PRINCIPALS],
        )
    else: raise
DATASET_ARN = resp['Arn']
print(f'\u2713 Inference dataset: {DATASET_ARN}')

Updating existing dataset...


✓ Inference dataset: arn:aws:quicksight:us-east-1:329430715989:dataset/fraud-governance-inference-dataset


### 6b. Create Drift Monitoring Dataset (monitoring_responses)

In [7]:
# monitoring_responses dataset — drift metrics from Evidently runs
drift_physical_table_map = {
    'monitoring-responses': {
        'RelationalTable': {
            'DataSourceArn': DATASOURCE_ARN,
            'Catalog': 'AwsDataCatalog',
            'Schema': ATHENA_DATABASE,
            'Name': ATHENA_MONITORING_RESPONSES_TABLE,
            'InputColumns': [
                {'Name': 'monitoring_run_id', 'Type': 'STRING'},
                {'Name': 'monitoring_timestamp', 'Type': 'DATETIME'},
                {'Name': 'endpoint_name', 'Type': 'STRING'},
                {'Name': 'model_version', 'Type': 'STRING'},
                {'Name': 'model_package_arn', 'Type': 'STRING'},
                {'Name': 'evaluation_snapshot_id', 'Type': 'STRING'},
                {'Name': 'data_drift_detected', 'Type': 'BIT'},
                {'Name': 'drifted_columns_count', 'Type': 'INTEGER'},
                {'Name': 'drifted_columns_share', 'Type': 'DECIMAL'},
                {'Name': 'features_analyzed', 'Type': 'INTEGER'},
                {'Name': 'data_sample_size', 'Type': 'INTEGER'},
                {'Name': 'model_drift_detected', 'Type': 'BIT'},
                {'Name': 'baseline_roc_auc', 'Type': 'DECIMAL'},
                {'Name': 'current_roc_auc', 'Type': 'DECIMAL'},
                {'Name': 'roc_auc_degradation', 'Type': 'DECIMAL'},
                {'Name': 'roc_auc_degradation_pct', 'Type': 'DECIMAL'},
                {'Name': 'accuracy', 'Type': 'DECIMAL'},
                {'Name': 'precision', 'Type': 'DECIMAL'},
                {'Name': 'recall', 'Type': 'DECIMAL'},
                {'Name': 'f1_score', 'Type': 'DECIMAL'},
                {'Name': 'model_sample_size', 'Type': 'INTEGER'},
                {'Name': 'per_feature_drift_scores', 'Type': 'STRING'},
                {'Name': 'evidently_report_s3_path', 'Type': 'STRING'},
                {'Name': 'mlflow_run_id', 'Type': 'STRING'},
                {'Name': 'alert_sent', 'Type': 'BIT'},
                {'Name': 'detection_engine', 'Type': 'STRING'},
                {'Name': 'created_at', 'Type': 'DATETIME'},
            ],
        }
    }
}

print()
print('=' * 70)
print('Dashboard build complete')
print('=' * 70)
print(f"QuickSight subscribed: {result['quicksight_subscribed']} (edition: {result['quicksight_edition']})")
print(f"Dashboard URL:  {result['dashboard_url']}")
print(f"Dashboard ARN:  {result['dashboard_arn']}")
print(f"Analysis ARN:   {result['analysis_arn']}")
print()
print('Datasets:')
for k in ('inference', 'drift', 'feature_drift', 'feature_level', 'accuracy'):
    print(f"  {k:14s} {result[f'{k}_dataset_arn']}")
if result.get('embed_url'):
    print(f"\nEmbed URL (valid 10h):\n  {result['embed_url']}")

Updating existing drift dataset...


✓ Drift dataset: arn:aws:quicksight:us-east-1:329430715989:dataset/fraud-governance-drift-dataset


## 3. Step-by-step (optional / for debugging)

The cells below reproduce `create_dashboard()` one section at a time by calling the
**same module functions** it uses internally. Use these when you want to re-run a
single stage (e.g. just recreate the Athena view, or just republish the dashboard)
without redoing everything. Skip this whole section if you already ran section 2.

Run the setup cell first to build the shared clients and resolve the QuickSight principals.

In [8]:
# Feature drift dataset — joins monitoring_responses to inference_responses
# by monitoring_run_id (foreign key the drift Lambda back-fills on each
# inference row it scored). Aggregates inference counts + avg fraud prob
# alongside the per-run drift metrics so dashboards can slice both sides
# of the drift↔inference relationship in a single dataset.

custom_sql = f'''
SELECT 
    m.monitoring_run_id,
    m.monitoring_timestamp,
    m.model_version,
    m.model_package_arn,
    m.evaluation_snapshot_id,
    m.drifted_columns_count,
    m.drifted_columns_share,
    m.features_analyzed,
    m.baseline_roc_auc,
    m.current_roc_auc,
    m.data_drift_detected,
    m.accuracy,
    m.precision,
    m.recall,
    m.f1_score,
    COUNT(DISTINCT i.inference_id) as inference_count,
    AVG(i.probability_positive) as avg_positive_prob,
    COUNT(CASE WHEN i.ground_truth IS NOT NULL THEN 1 END) as gt_count,
    MIN(i.request_timestamp) as window_start,
    MAX(i.request_timestamp) as window_end
FROM {ATHENA_DATABASE}.{ATHENA_MONITORING_RESPONSES_TABLE} m
LEFT JOIN {ATHENA_DATABASE}.{ATHENA_INFERENCE_TABLE} i
    -- Exact foreign-key join: the drift Lambda back-fills monitoring_run_id
    -- on inference_responses for every row it scored. Older versions used a
    -- 24-hour time-window approximation here, which missed rows when runs
    -- were >24h apart and double-counted when runs overlapped.
    ON i.monitoring_run_id = m.monitoring_run_id
GROUP BY 1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
ORDER BY m.monitoring_timestamp DESC
'''

feature_drift_physical_table_map = {
    'feature-drift-joined': {
        'CustomSql': {
            'DataSourceArn': DATASOURCE_ARN,
            'Name': 'FeatureDriftJoin',
            'SqlQuery': custom_sql,
            'Columns': [
                {'Name': 'monitoring_run_id', 'Type': 'STRING'},
                {'Name': 'monitoring_timestamp', 'Type': 'DATETIME'},
                {'Name': 'model_version', 'Type': 'STRING'},
                {'Name': 'model_package_arn', 'Type': 'STRING'},
                {'Name': 'evaluation_snapshot_id', 'Type': 'STRING'},
                {'Name': 'drifted_columns_count', 'Type': 'INTEGER'},
                {'Name': 'drifted_columns_share', 'Type': 'DECIMAL'},
                {'Name': 'features_analyzed', 'Type': 'INTEGER'},
                {'Name': 'baseline_roc_auc', 'Type': 'DECIMAL'},
                {'Name': 'current_roc_auc', 'Type': 'DECIMAL'},
                {'Name': 'data_drift_detected', 'Type': 'BIT'},
                {'Name': 'accuracy', 'Type': 'DECIMAL'},
                {'Name': 'precision', 'Type': 'DECIMAL'},
                {'Name': 'recall', 'Type': 'DECIMAL'},
                {'Name': 'f1_score', 'Type': 'DECIMAL'},
                {'Name': 'inference_count', 'Type': 'INTEGER'},
                {'Name': 'avg_positive_prob', 'Type': 'DECIMAL'},
                {'Name': 'gt_count', 'Type': 'INTEGER'},
                # Earliest / latest request_timestamp seen for this drift run.
                # Lets visuals show "this run scored inferences from <start> to <end>".
                {'Name': 'window_start', 'Type': 'DATETIME'},
                {'Name': 'window_end', 'Type': 'DATETIME'},
            ],
        }
    }
}

feature_drift_logical_table_map = {
    'feature-drift-logical': {
        'Alias': 'Feature Drift Analysis',
        'Source': {'PhysicalTableId': 'feature-drift-joined'},
    }
}

feature_drift_dset_common = dict(
    AwsAccountId=ACCOUNT_ID, DataSetId=QUICKSIGHT_FEATURE_DRIFT_DATASET_ID,
    Name=QUICKSIGHT_FEATURE_DRIFT_DATASET_NAME,
    PhysicalTableMap=feature_drift_physical_table_map,
    LogicalTableMap=feature_drift_logical_table_map,
    ImportMode='DIRECT_QUERY',
)

try:
    quicksight.describe_data_set(AwsAccountId=ACCOUNT_ID, DataSetId=QUICKSIGHT_FEATURE_DRIFT_DATASET_ID)
    print('Updating existing feature drift dataset...')
    resp = quicksight.update_data_set(**feature_drift_dset_common)
except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceNotFoundException':
        print('Creating new feature drift dataset...')
        resp = quicksight.create_data_set(
            **feature_drift_dset_common,
            Permissions=[{'Principal': p, 'Actions': DSET_ACTIONS} for p in QS_PRINCIPALS],
        )
    else: raise

FEATURE_DRIFT_DATASET_ARN = resp['Arn']
print(f'✓ Feature drift dataset: {FEATURE_DRIFT_DATASET_ARN}')

Updating existing feature drift dataset...


✓ Feature drift dataset: arn:aws:quicksight:us-east-1:329430715989:dataset/fraud-governance-feature-drift-dataset


In [9]:
# Create Athena View for Feature-Level Drift Analysis
# This view unpacks the JSON per_feature_drift_scores into individual rows

import time

print("Creating feature_drift_detail view...")

create_view_sql = f"""
CREATE OR REPLACE VIEW {ATHENA_DATABASE}.feature_drift_detail AS
SELECT
    monitoring_run_id,
    monitoring_timestamp,
    model_version,
    model_package_arn,
    evaluation_snapshot_id,
    endpoint_name,
    data_drift_detected,
    drifted_columns_count,
    drifted_columns_share,
    baseline_roc_auc,
    current_roc_auc,
    feature_name,                    -- Unpacked from JSON
    drift_score,                     -- Unpacked from JSON
    CASE
        WHEN drift_score > 0.25 THEN 'Significant'
        WHEN drift_score > 0.1 THEN 'Moderate'
        ELSE 'Low'
    END as drift_severity,           -- Computed severity
    CASE WHEN drift_score > 0.1 THEN true ELSE false END as drift_detected
FROM {ATHENA_DATABASE}.monitoring_responses
CROSS JOIN UNNEST(
    CAST(json_parse(per_feature_drift_scores) AS MAP(VARCHAR, DOUBLE))
) AS t(feature_name, drift_score)
WHERE per_feature_drift_scores IS NOT NULL
    AND per_feature_drift_scores != 'null'
    AND per_feature_drift_scores != '{{}}'
"""

# Execute view creation
response = athena.start_query_execution(
    QueryString=create_view_sql,
    QueryExecutionContext={'Database': ATHENA_DATABASE},
    ResultConfiguration={'OutputLocation': ATHENA_OUTPUT_S3}
)

query_id = response['QueryExecutionId']
print(f"Query execution ID: {query_id}")

# Wait for completion
while True:
    result = athena.get_query_execution(QueryExecutionId=query_id)
    status = result['QueryExecution']['Status']['State']
    if status == 'SUCCEEDED':
        print("✓ View created successfully!")
        break
    elif status in ['FAILED', 'CANCELLED']:
        reason = result['QueryExecution']['Status'].get('StateChangeReason', 'Unknown')
        print(f"✗ Failed: {reason}")
        raise Exception(f"View creation failed: {reason}")
    time.sleep(2)

# Test the view
print("\nTesting view with sample query...")
test_query = f"""
SELECT
    COUNT(*) as total_rows,
    COUNT(DISTINCT feature_name) as features,
    COUNT(DISTINCT monitoring_run_id) as runs,
    MIN(monitoring_timestamp) as first_run,
    MAX(monitoring_timestamp) as last_run
FROM {ATHENA_DATABASE}.feature_drift_detail
"""

response = athena.start_query_execution(
    QueryString=test_query,
    QueryExecutionContext={'Database': ATHENA_DATABASE},
    ResultConfiguration={'OutputLocation': ATHENA_OUTPUT_S3}
)

query_id = response['QueryExecutionId']
while True:
    result = athena.get_query_execution(QueryExecutionId=query_id)
    status = result['QueryExecution']['Status']['State']
    if status == 'SUCCEEDED':
        results = athena.get_query_results(QueryExecutionId=query_id)
        if len(results['ResultSet']['Rows']) > 1:
            data = results['ResultSet']['Rows'][1]['Data']
            print(f"✓ View test successful!")
            print(f"  Total rows: {data[0].get('VarCharValue', '0')}")
            print(f"  Unique features: {data[1].get('VarCharValue', '0')}")
            print(f"  Monitoring runs: {data[2].get('VarCharValue', '0')}")
            print(f"  First run: {data[3].get('VarCharValue', 'N/A')}")
            print(f"  Last run: {data[4].get('VarCharValue', 'N/A')}")
        break
    elif status in ['FAILED', 'CANCELLED']:
        reason = result['QueryExecution']['Status'].get('StateChangeReason', 'Unknown')
        print(f"✗ Test query failed: {reason}")
        break
    time.sleep(2)

print("\n✓ View is ready for QuickSight dataset!")

Creating feature_drift_detail view...
Query execution ID: 16a6c3ce-35dc-4827-8930-f6fd339913ba


✓ View created successfully!

Testing view with sample query...


✓ View test successful!
  Total rows: 40
  Unique features: 20
  Monitoring runs: 5
  First run: 2026-07-16 17:01:47.000000
  Last run: 2026-07-16 20:06:00.000000

✓ View is ready for QuickSight dataset!


# Grant Lake Formation permissions on the feature_drift_detail VIEW
# Views require separate permissions from underlying tables

import subprocess
import json

print("Granting Lake Formation permissions on feature_drift_detail view...")

# Grant to IAM_ALLOWED_PRINCIPALS (allows any IAM role with Athena/Glue permissions)
resource_json = json.dumps({
    'Table': {
        'DatabaseName': ATHENA_DATABASE,
        'Name': 'feature_drift_detail'
    }
})

cmd = [
    'aws', 'lakeformation', 'grant-permissions',
    '--principal', '{"DataLakePrincipalIdentifier":"IAM_ALLOWED_PRINCIPALS"}',
    '--resource', resource_json,
    '--permissions', 'SELECT', 'DESCRIBE'
]

try:
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
    if result.returncode == 0:
        print("✓ Lake Formation permissions granted on view")
        print("  Principal: IAM_ALLOWED_PRINCIPALS")
        print("  Resource: feature_drift_detail")
        print("  Permissions: SELECT, DESCRIBE")
    else:
        # Check if already granted
        if 'AlreadyExistsException' in result.stderr or 'already exists' in result.stderr.lower():
            print("✓ Permissions already exist (no change needed)")
        else:
            print(f"⚠️  Warning: {result.stderr}")
            print("  This might be OK if permissions were granted previously")
except subprocess.TimeoutExpired:
    print("⚠️  Command timed out (permissions may still be granted)")
except Exception as e:
    print(f"⚠️  Error: {e}")
    print("  You may need to grant permissions manually in Lake Formation console")

print("\n✓ Ready for QuickSight dataset creation!")

In [11]:
DATASOURCE_ARN = gov.create_or_update_datasource(
    account_id=ACCOUNT_ID, quicksight_principals=QS_PRINCIPALS, quicksight_client=quicksight,
)

try:
    quicksight.describe_data_set(AwsAccountId=ACCOUNT_ID, DataSetId=FEATURE_LEVEL_DATASET_ID)
    print('Updating existing feature-level dataset...')
    resp = quicksight.update_data_set(**feature_level_dset_common)
except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceNotFoundException':
        print('Creating new feature-level dataset...')
        resp = quicksight.create_data_set(
            **feature_level_dset_common,
            Permissions=[{'Principal': p, 'Actions': DSET_ACTIONS} for p in QS_PRINCIPALS],
        )
    else:
        raise

FEATURE_LEVEL_DATASET_ARN = resp['Arn']
print(f'✓ Feature-level dataset: {FEATURE_LEVEL_DATASET_ARN}')

Updating existing feature-level dataset...


✓ Feature-level dataset: arn:aws:quicksight:us-east-1:329430715989:dataset/fraud-governance-feature-level-dataset


### 3b. Datasets + feature_drift_detail view

Order matters: the `feature_drift_detail` view must exist before the feature-level
dataset references it. The view parses the per-feature JSON
(`{score, magnitude, method, threshold}`) and exposes `drift_magnitude` / `drift_method`.

In [12]:
# Dataset for prediction accuracy (joins inference with ground truth)
ACCURACY_DATASET_ID = f'{DATASET_ID}-accuracy'
ACCURACY_DATASET_ARN = f'arn:aws:quicksight:{AWS_DEFAULT_REGION}:{ACCOUNT_ID}:dataset/{ACCURACY_DATASET_ID}'
ACCURACY_DATASET_NAME = 'Prediction Accuracy Timeline'

# Use CustomSQL to join inference_responses with ground_truth_updates
accuracy_custom_sql = f"""
SELECT 
    DATE(i.request_timestamp) as inference_date,
    i.inference_id,
    i.endpoint_name,
    i.model_version,
    i.prediction as predicted_fraud,
    CAST(g.actual_fraud AS INT) as actual_fraud,
    CASE 
        WHEN i.prediction = CAST(g.actual_fraud AS INT) THEN 1 
        ELSE 0 
    END as prediction_match,
    CASE
        WHEN i.prediction = 1 AND CAST(g.actual_fraud AS INT) = 1 THEN 'True Positive'
        WHEN i.prediction = 0 AND CAST(g.actual_fraud AS INT) = 0 THEN 'True Negative'
        WHEN i.prediction = 1 AND CAST(g.actual_fraud AS INT) = 0 THEN 'False Positive'
        WHEN i.prediction = 0 AND CAST(g.actual_fraud AS INT) = 1 THEN 'False Negative'
        ELSE 'Unknown'
    END as prediction_category,
    i.request_timestamp as prediction_time,
    g.confirmation_timestamp as ground_truth_time,
    g.days_since_prediction
FROM {ATHENA_DATABASE}.inference_responses i
INNER JOIN {ATHENA_DATABASE}.ground_truth_updates g
    ON i.inference_id = g.inference_id
WHERE i.request_timestamp >= CURRENT_DATE - INTERVAL '30' DAY
    AND g.actual_fraud IS NOT NULL
ORDER BY i.request_timestamp DESC
"""

accuracy_physical_table = {
    'accuracy-join': {
        'CustomSql': {
            'DataSourceArn': DATASOURCE_ARN,
            'Name': 'accuracy-join',
            'SqlQuery': accuracy_custom_sql,
            'Columns': [
                {'Name': 'inference_date', 'Type': 'DATETIME'},
                {'Name': 'inference_id', 'Type': 'STRING'},
                {'Name': 'endpoint_name', 'Type': 'STRING'},
                {'Name': 'model_version', 'Type': 'STRING'},
                {'Name': 'predicted_fraud', 'Type': 'INTEGER'},
                {'Name': 'actual_fraud', 'Type': 'INTEGER'},
                {'Name': 'prediction_match', 'Type': 'INTEGER'},
                {'Name': 'prediction_category', 'Type': 'STRING'},
                {'Name': 'prediction_time', 'Type': 'DATETIME'},
                {'Name': 'ground_truth_time', 'Type': 'DATETIME'},
                {'Name': 'days_since_prediction', 'Type': 'DECIMAL'},
            ]
        }
    }
}

# Calculated field for accuracy percentage
accuracy_logical_table = {
    'accuracy-logical': {
        'Alias': 'Prediction Accuracy',
        'Source': {'PhysicalTableId': 'accuracy-join'},
        'DataTransforms': [
            {
                'CreateColumnsOperation': {
                    'Columns': [
                        {
                            'ColumnName': 'accuracy_pct',
                            'ColumnId': 'accuracy-pct',
                            'Expression': 'sum({prediction_match}) / count({inference_id}) * 100'
                        },
                        {
                            'ColumnName': 'is_correct',
                            'ColumnId': 'is-correct',
                            'Expression': 'ifelse({prediction_match} = 1, "Correct", "Incorrect")'
                        }
                    ]
                }
            }
        ]
    }
}

# Create dataset
try:
    quicksight.describe_data_set(AwsAccountId=ACCOUNT_ID, DataSetId=ACCURACY_DATASET_ID)
    print(f'Dataset {ACCURACY_DATASET_ID} already exists, updating...')
    resp = quicksight.update_data_set(
        AwsAccountId=ACCOUNT_ID,
        DataSetId=ACCURACY_DATASET_ID,
        Name=ACCURACY_DATASET_NAME,
        PhysicalTableMap=accuracy_physical_table,
        LogicalTableMap=accuracy_logical_table,
        ImportMode='DIRECT_QUERY',
    )
except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceNotFoundException':
        print(f'Creating new dataset {ACCURACY_DATASET_ID}...')
        resp = quicksight.create_data_set(
            AwsAccountId=ACCOUNT_ID,
            DataSetId=ACCURACY_DATASET_ID,
            Name=ACCURACY_DATASET_NAME,
            PhysicalTableMap=accuracy_physical_table,
            LogicalTableMap=accuracy_logical_table,
            ImportMode='DIRECT_QUERY',
            Permissions=[{'Principal': p, 'Actions': DSET_ACTIONS} for p in QS_PRINCIPALS],
        )
        print('✅ Dataset created')
    else:
        raise

ACCURACY_DATASET_ARN = resp['Arn']

print(f'Dataset ARN: {ACCURACY_DATASET_ARN}')

Dataset fraud-governance-inference-dataset-accuracy already exists, updating...


Dataset ARN: arn:aws:quicksight:us-east-1:329430715989:dataset/fraud-governance-inference-dataset-accuracy


## 7. Define Visuals

**Sheet 1 — Inference Monitoring** (6 visuals from `inference_responses`):
1. Prediction Volume Over Time (line chart)
2. Fraud Probability Distribution (histogram)
3. Prediction Accuracy Breakdown (donut)
4. Risk Tier Distribution (bar chart)
5. Inference Latency Trend (line chart)
6. Total Inferences KPI

**Sheet 2 — Drift Trend Analysis** (6 visuals from `monitoring_responses`):
7. Data Drift Share Over Time
8. Drifted Columns Count Trend
9. ROC-AUC Degradation Trend (baseline vs current)
10. Model Performance Metrics Over Time (accuracy, precision, recall, F1)
11. Drift Alerts Timeline
12. Latest Drift Share KPI

In [13]:
# Helper: column identifier shorthand
def col(name):
    return {'DataSetIdentifier': 'inference-ds', 'ColumnName': name}

# V1: Prediction Volume Over Time (line chart)
v1_volume = {
    'LineChartVisual': {
        'VisualId': 'v1-prediction-volume',
        'Title': {'Visibility': 'VISIBLE', 'FormatText': {'PlainText': 'Prediction Volume Over Time'}},
        'ChartConfiguration': {
            'FieldWells': {
                'LineChartAggregatedFieldWells': {
                    'Category': [{'DateDimensionField': {'FieldId': 'date-dim', 'Column': col('request_timestamp'), 'DateGranularity': 'DAY'}}],
                    'Values': [{'NumericalMeasureField': {'FieldId': 'count-id', 'Column': col('probability_positive'), 'AggregationFunction': {'SimpleNumericalAggregation': 'COUNT'}}}],
                    'Colors': [],
                }
            }
        }
    }
}

# V2: Fraud Probability Distribution (histogram)
v2_histogram = {
    'HistogramVisual': {
        'VisualId': 'v2-fraud-prob-dist',
        'Title': {'Visibility': 'VISIBLE', 'FormatText': {'PlainText': 'Fraud Probability Distribution'}},
        'ChartConfiguration': {
            'FieldWells': {
                'HistogramAggregatedFieldWells': {
                    'Values': [{'NumericalMeasureField': {'FieldId': 'prob-fraud', 'Column': col('probability_positive')}}],
                }
            },
            'BinOptions': {'BinCount': {'Value': 20}},
        }
    }
}

# V3: Prediction Accuracy (pie/donut chart)
v3_accuracy = {
    'PieChartVisual': {
        'VisualId': 'v3-prediction-accuracy',
        'Title': {'Visibility': 'VISIBLE', 'FormatText': {'PlainText': 'Prediction Accuracy Breakdown'}},
        'ChartConfiguration': {
            'FieldWells': {
                'PieChartAggregatedFieldWells': {
                    'Category': [{'CategoricalDimensionField': {'FieldId': 'acc-cat', 'Column': col('prediction_accuracy')}}],
                    'Values': [{'NumericalMeasureField': {'FieldId': 'acc-count', 'Column': col('probability_positive'), 'AggregationFunction': {'SimpleNumericalAggregation': 'COUNT'}}}],
                }
            },
            'DonutOptions': {'ArcOptions': {'ArcThickness': 'MEDIUM'}},
        }
    }
}

# V4: Risk Tier Distribution (bar chart)
v4_risk = {
    'BarChartVisual': {
        'VisualId': 'v4-risk-tier',
        'Title': {'Visibility': 'VISIBLE', 'FormatText': {'PlainText': 'Risk Tier Distribution'}},
        'ChartConfiguration': {
            'FieldWells': {
                'BarChartAggregatedFieldWells': {
                    'Category': [{'CategoricalDimensionField': {'FieldId': 'risk-cat', 'Column': col('risk_tier')}}],
                    'Values': [{'NumericalMeasureField': {'FieldId': 'risk-count', 'Column': col('probability_positive'), 'AggregationFunction': {'SimpleNumericalAggregation': 'COUNT'}}}],
                    'Colors': [],
                }
            },
            'Orientation': 'VERTICAL',
        }
    }
}

# V5: Inference Latency Trend (line chart) — uses inference_latency_ms
v5_latency = {
    'LineChartVisual': {
        'VisualId': 'v5-latency-trend',
        'Title': {'Visibility': 'VISIBLE', 'FormatText': {'PlainText': 'Inference Latency Trend (ms)'}},
        'ChartConfiguration': {
            'FieldWells': {
                'LineChartAggregatedFieldWells': {
                    'Category': [{'DateDimensionField': {'FieldId': 'lat-date', 'Column': col('request_timestamp'), 'DateGranularity': 'DAY'}}],
                    'Values': [{'NumericalMeasureField': {'FieldId': 'avg-latency', 'Column': col('inference_latency_ms'), 'AggregationFunction': {'SimpleNumericalAggregation': 'AVERAGE'}}}],
                    'Colors': [],
                }
            }
        }
    }
}

# V6: KPI — Total Inferences
v6_kpi = {
    'KPIVisual': {
        'VisualId': 'v6-total-inferences',
        'Title': {'Visibility': 'VISIBLE', 'FormatText': {'PlainText': 'Total Inferences'}},
        'ChartConfiguration': {
            'FieldWells': {
                'Values': [{'NumericalMeasureField': {'FieldId': 'kpi-count', 'Column': col('probability_positive'), 'AggregationFunction': {'SimpleNumericalAggregation': 'COUNT'}}}],
            }
        }
    }
}

# Helper function for accuracy dataset columns
def acol(name):
    return {'DataSetIdentifier': 'accuracy-ds', 'ColumnName': name}

# V7: Daily Accuracy Line Chart
v7_accuracy_trend = {
    'LineChartVisual': {
        'VisualId': 'v7-accuracy-trend',
        'Title': {'Visibility': 'VISIBLE', 'FormatText': {'PlainText': 'Correct Predictions Over Time (Count)'}},
        'ChartConfiguration': {
            'FieldWells': {
                'LineChartAggregatedFieldWells': {
                    'Category': [{'DateDimensionField': {'FieldId': 'date', 'Column': acol('inference_date'), 'DateGranularity': 'DAY'}}],
                    'Values': [
                        {
                            'NumericalMeasureField': {
                                'FieldId': 'match-sum',
                                'Column': acol('prediction_match'),
                                'AggregationFunction': {'SimpleNumericalAggregation': 'SUM'}
                            }
                        }
                    ]
                }
            },
            'Type': 'LINE',
            'Legend': {'Visibility': 'HIDDEN'},
            'DataLabels': {'Visibility': 'HIDDEN'},
            'Tooltip': {'TooltipVisibility': 'VISIBLE'}
        },
        'Actions': []
    }
}

# V8: Correct vs Incorrect Stacked Bar Chart
v8_correct_incorrect = {
    'BarChartVisual': {
        'VisualId': 'v8-correct-incorrect',
        'Title': {'Visibility': 'VISIBLE', 'FormatText': {'PlainText': 'Daily Predictions: Correct vs Incorrect'}},
        'ChartConfiguration': {
            'FieldWells': {
                'BarChartAggregatedFieldWells': {
                    'Category': [{'DateDimensionField': {'FieldId': 'date', 'Column': acol('inference_date'), 'DateGranularity': 'DAY'}}],
                    'Values': [
                        {
                            'CategoricalMeasureField': {
                                'FieldId': 'count',
                                'Column': acol('inference_id'),
                                'AggregationFunction': 'COUNT'
                            }
                        }
                    ],
                    'Colors': [{'CategoricalDimensionField': {'FieldId': 'category', 'Column': acol('is_correct')}}]
                }
            },
            'Orientation': 'VERTICAL',
            'BarsArrangement': 'STACKED',
            'DataLabels': {'Visibility': 'HIDDEN'},
            'Legend': {'Visibility': 'VISIBLE', 'Position': 'RIGHT'},
            'Tooltip': {'TooltipVisibility': 'VISIBLE'}
        },
        'Actions': []
    }
}



INFERENCE_VISUALS = [v1_volume, v2_histogram, v3_accuracy, v4_risk, v5_latency, v6_kpi, v7_accuracy_trend, v8_correct_incorrect]
print(f'Defined {len(INFERENCE_VISUALS)} inference visuals')


Defined 8 inference visuals


### 7b. Drift Trend Analysis Visuals (Sheet 2)

In [14]:
inference_arn     = gov.create_inference_dataset(DATASOURCE_ARN, ACCOUNT_ID, QS_PRINCIPALS, quicksight_client=quicksight)
drift_arn         = gov.create_drift_dataset(DATASOURCE_ARN, ACCOUNT_ID, QS_PRINCIPALS, quicksight_client=quicksight)
feature_drift_arn = gov.create_feature_drift_dataset(DATASOURCE_ARN, ACCOUNT_ID, QS_PRINCIPALS, quicksight_client=quicksight)

# Create/replace the Athena view, grant it, then the dataset that reads it.
gov.create_feature_drift_detail_view(athena_client=athena)
gov.grant_feature_drift_view_permissions(region=AWS_DEFAULT_REGION)
feature_level_arn = gov.create_feature_level_dataset(DATASOURCE_ARN, ACCOUNT_ID, QS_PRINCIPALS, quicksight_client=quicksight)

accuracy_arn = gov.create_accuracy_dataset(DATASOURCE_ARN, ACCOUNT_ID, QS_PRINCIPALS, quicksight_client=quicksight)

DATASET_ARNS = {
    'inference': inference_arn,
    'drift': drift_arn,
    'feature_drift': feature_drift_arn,
    'feature_level': feature_level_arn,
    'accuracy': accuracy_arn,
}
DATASET_ARNS

### 3c. Analysis + dashboard

The visual definitions (Model Drift, Data Drift, and Feature Drift sheets — the last
built by `build_feature_drift_visuals()` using `drift_magnitude`) are assembled inside
these calls from the module. Nothing is defined in the notebook.

In [16]:
# Helper: column identifier for feature-level dataset
def flcol(name):
    return {'DataSetIdentifier': 'feature-level-ds', 'ColumnName': name}

# V17: Feature Drift Timeline (line chart - drift scores over time by feature)
v17_feature_timeline = {
    'LineChartVisual': {
        'VisualId': 'v17-feature-timeline',
        'Title': {'Visibility': 'VISIBLE', 'FormatText': {'PlainText': 'Feature Drift Score Timeline'}},
        'ChartConfiguration': {
            'FieldWells': {
                'LineChartAggregatedFieldWells': {
                    'Category': [{'DateDimensionField': {'FieldId': 'time', 'Column': flcol('monitoring_timestamp'), 'DateGranularity': 'DAY'}}],
                    'Values': [{'NumericalMeasureField': {'FieldId': 'drift', 'Column': flcol('drift_score'), 'AggregationFunction': {'SimpleNumericalAggregation': 'AVERAGE'}}}],
                    'Colors': [{'CategoricalDimensionField': {'FieldId': 'feature', 'Column': flcol('feature_name')}}],
                }
            },
            'SortConfiguration': {},
            'Type': 'LINE',
            'Legend': {'Visibility': 'VISIBLE', 'Position': 'RIGHT'},
            'PrimaryYAxisDisplayOptions': {'AxisOptions': {'AxisLineVisibility': 'VISIBLE'}},
        }
    }
}

# V18: Top Drifting Features (horizontal bar - avg drift by feature)
v18_top_features = {
    'BarChartVisual': {
        'VisualId': 'v18-top-features',
        'Title': {'Visibility': 'VISIBLE', 'FormatText': {'PlainText': 'Top 15 Drifting Features'}},
        'ChartConfiguration': {
            'FieldWells': {
                'BarChartAggregatedFieldWells': {
                    'Category': [{'CategoricalDimensionField': {'FieldId': 'feature', 'Column': flcol('feature_name')}}],
                    'Values': [{'NumericalMeasureField': {'FieldId': 'drift', 'Column': flcol('drift_score'), 'AggregationFunction': {'SimpleNumericalAggregation': 'AVERAGE'}}}],
                }
            },
            'SortConfiguration': {
                'CategoryItemsLimit': {'OtherCategories': 'INCLUDE', 'ItemsLimit': 15},
                'CategorySort': [{'FieldSort': {'FieldId': 'drift', 'Direction': 'DESC'}}],
            },
            'Orientation': 'HORIZONTAL',
            'BarsArrangement': 'CLUSTERED',
            'Legend': {'Visibility': 'HIDDEN'},
        }
    }
}

# V19: Feature Drift Detail Table
v19_feature_table = {
    'TableVisual': {
        'VisualId': 'v19-feature-table',
        'Title': {'Visibility': 'VISIBLE', 'FormatText': {'PlainText': 'Feature Drift Details'}},
        'ChartConfiguration': {
            'FieldWells': {
                'TableAggregatedFieldWells': {
                    'GroupBy': [
                        {'DateDimensionField': {'FieldId': 'time', 'Column': flcol('monitoring_timestamp'), 'DateGranularity': 'DAY'}},
                        {'CategoricalDimensionField': {'FieldId': 'feature', 'Column': flcol('feature_name')}},
                        {'CategoricalDimensionField': {'FieldId': 'severity', 'Column': flcol('drift_severity')}},
                        {'CategoricalDimensionField': {'FieldId': 'model', 'Column': flcol('model_version')}},
                    ],
                    'Values': [
                        {'NumericalMeasureField': {'FieldId': 'drift', 'Column': flcol('drift_score'), 'AggregationFunction': {'SimpleNumericalAggregation': 'AVERAGE'}}},
                    ],
                }
            },
            'SortConfiguration': {
                'RowSort': [{'FieldSort': {'FieldId': 'time', 'Direction': 'DESC'}}],
            },
        }
    }
}

# V20: Drift Severity Distribution (stacked bar by severity)
v20_severity_dist = {
    'BarChartVisual': {
        'VisualId': 'v20-severity-dist',
        'Title': {'Visibility': 'VISIBLE', 'FormatText': {'PlainText': 'Drift Severity by Feature (Top 15)'}},
        'ChartConfiguration': {
            'FieldWells': {
                'BarChartAggregatedFieldWells': {
                    'Category': [{'CategoricalDimensionField': {'FieldId': 'feature', 'Column': flcol('feature_name')}}],
                    'Values': [{'NumericalMeasureField': {'FieldId': 'count', 'Column': flcol('drift_score'), 'AggregationFunction': {'SimpleNumericalAggregation': 'COUNT'}}}],
                    'Colors': [{'CategoricalDimensionField': {'FieldId': 'severity', 'Column': flcol('drift_severity')}}],
                }
            },
            'SortConfiguration': {
                'CategoryItemsLimit': {'OtherCategories': 'INCLUDE', 'ItemsLimit': 15},
                'CategorySort': [{'FieldSort': {'FieldId': 'count', 'Direction': 'DESC'}}],
            },
            'Orientation': 'HORIZONTAL',
            'BarsArrangement': 'STACKED',
            'Legend': {'Visibility': 'VISIBLE', 'Position': 'RIGHT'},
        }
    }
}

# V21: Highest Drifting Feature KPI
v21_worst_feature = {
    'KPIVisual': {
        'VisualId': 'v21-worst-feature',
        'Title': {'Visibility': 'VISIBLE', 'FormatText': {'PlainText': 'Highest Drift Score'}},
        'ChartConfiguration': {
            'FieldWells': {
                'Values': [{'NumericalMeasureField': {'FieldId': 'max-drift', 'Column': flcol('drift_score'), 'AggregationFunction': {'SimpleNumericalAggregation': 'MAX'}}}],
            },
            'SortConfiguration': {},
        }
    }
}

# V22: Feature Drift Heatmap (pivot table)
v22_drift_heatmap = {
    'PivotTableVisual': {
        'VisualId': 'v22-drift-heatmap',
        'Title': {'Visibility': 'VISIBLE', 'FormatText': {'PlainText': 'Feature Drift Heatmap (Features × Time)'}},
        'ChartConfiguration': {
            'FieldWells': {
                'PivotTableAggregatedFieldWells': {
                    'Rows': [{'CategoricalDimensionField': {'FieldId': 'feature', 'Column': flcol('feature_name')}}],
                    'Columns': [{'DateDimensionField': {'FieldId': 'date', 'Column': flcol('monitoring_timestamp'), 'DateGranularity': 'DAY'}}],
                    'Values': [{'NumericalMeasureField': {'FieldId': 'drift', 'Column': flcol('drift_score'), 'AggregationFunction': {'SimpleNumericalAggregation': 'AVERAGE'}}}],
                }
            },
            'SortConfiguration': {},
        }
    }
}

FEATURE_LEVEL_VISUALS = [
    v17_feature_timeline,
    v18_top_features,
    v19_feature_table,
    v20_severity_dist,
    v21_worst_feature,
    v22_drift_heatmap
]

print(f'Defined {len(FEATURE_LEVEL_VISUALS)} feature-level visuals')
print('  V17: Feature Drift Score Timeline (line chart with line per feature)')
print('  V18: Top 15 Drifting Features (horizontal bar)')
print('  V19: Feature Drift Details (table)')
print('  V20: Drift Severity by Feature (stacked bar)')
print('  V21: Highest Drift Score (KPI)')
print('  V22: Feature Drift Heatmap (pivot table)')


Defined 6 feature-level visuals
  V17: Feature Drift Score Timeline (line chart with line per feature)
  V18: Top 15 Drifting Features (horizontal bar)
  V19: Feature Drift Details (table)
  V20: Drift Severity by Feature (stacked bar)
  V21: Highest Drift Score (KPI)
  V22: Feature Drift Heatmap (pivot table)


In [17]:
# VERIFICATION: Check visual structure before creating analysis
import json

def verify_visuals():
    """Verify DRIFT_VISUALS and FEATURE_DRIFT_VISUALS have correct structure"""
    issues = []
    
    # Check DRIFT_VISUALS
    for i, visual in enumerate(DRIFT_VISUALS):
        if 'LineChartVisual' in visual:
            config = visual['LineChartVisual']['ChartConfiguration']
            if 'PrimaryYAxisDisplayOptions' in config:
                opts = config['PrimaryYAxisDisplayOptions']
                if 'AxisLineVisibility' in opts:
                    issues.append(f"DRIFT_VISUALS[{i}]: AxisLineVisibility at top level (needs AxisOptions wrapper)")
                elif 'AxisOptions' not in opts:
                    issues.append(f"DRIFT_VISUALS[{i}]: Missing AxisOptions wrapper")
    
    # Check FEATURE_DRIFT_VISUALS
    for i, visual in enumerate(FEATURE_DRIFT_VISUALS):
        if 'LineChartVisual' in visual:
            config = visual['LineChartVisual']['ChartConfiguration']
            if 'PrimaryYAxisDisplayOptions' in config:
                opts = config['PrimaryYAxisDisplayOptions']
                if 'AxisLineVisibility' in opts:
                    issues.append(f"FEATURE_DRIFT_VISUALS[{i}]: AxisLineVisibility at top level")
                elif 'AxisOptions' not in opts:
                    issues.append(f"FEATURE_DRIFT_VISUALS[{i}]: Missing AxisOptions wrapper")
        
        # Check DateMeasureField
        if 'TableVisual' in visual:
            fw = visual['TableVisual']['ChartConfiguration']['FieldWells']['TableAggregatedFieldWells']
            for val in fw.get('Values', []):
                if 'DateMeasureField' in val:
                    agg = val['DateMeasureField']['AggregationFunction']
                    if isinstance(agg, dict):
                        issues.append(f"FEATURE_DRIFT_VISUALS[{i}]: DateMeasureField AggregationFunction is dict (should be string)")
    
    if issues:
        print("❌ VALIDATION FAILED - Variables have old structure:")
        for issue in issues:
            print(f"  - {issue}")
        print("\n⚠️  You MUST re-run cells 20 and 22 to fix this!")
        print("    The notebook file is correct, but your Python variables are outdated.")
        return False
    else:
        print("✅ VALIDATION PASSED")
        print(f"  - DRIFT_VISUALS: {len(DRIFT_VISUALS)} visuals with correct structure")
        print(f"  - FEATURE_DRIFT_VISUALS: {len(FEATURE_DRIFT_VISUALS)} visuals with correct structure")
        print("\n✓ Safe to proceed with analysis creation (next cell)")
        return True

verify_visuals()

✅ VALIDATION PASSED
  - DRIFT_VISUALS: 6 visuals with correct structure
  - FEATURE_DRIFT_VISUALS: 4 visuals with correct structure

✓ Safe to proceed with analysis creation (next cell)


True

In [18]:
import json
print("inference_id in INFERENCE_VISUALS:",
        "'inference_id'" in json.dumps(INFERENCE_VISUALS))
print("probability_fraud in INFERENCE_VISUALS:",
        "'probability_positive'" in json.dumps(INFERENCE_VISUALS))

inference_id in INFERENCE_VISUALS: False
probability_fraud in INFERENCE_VISUALS: False


## 4. Generate Embed URL (optional)

In [ ]:
# ⚠️ DIAGNOSTIC: Check if variables have been updated
# Run this BEFORE Cell 25 to verify you've re-run cells 18, 20, 22

import json

print("=" * 80)
print("CHECKING VARIABLE VALUES IN KERNEL MEMORY")
print("=" * 80)

issues = []

# Check INFERENCE_VISUALS
inference_str = json.dumps(INFERENCE_VISUALS)
if "'inference_id'" in inference_str:
    issues.append("❌ INFERENCE_VISUALS still uses 'inference_id' (Cell 18 NOT re-run)")
elif "'probability_positive'" in inference_str:
    print("✅ INFERENCE_VISUALS: Uses probability_fraud (Cell 18 was re-run)")

# Check DRIFT_VISUALS
drift_str = json.dumps(DRIFT_VISUALS)
if "'monitoring_run_id'" in drift_str:
    issues.append("❌ DRIFT_VISUALS still uses 'monitoring_run_id' (Cell 20 NOT re-run)")
elif "'drifted_columns_count'" in drift_str:
    print("✅ DRIFT_VISUALS: Uses drifted_columns_count (Cell 20 was re-run)")

# Check FEATURE_DRIFT_VISUALS
feature_str = json.dumps(FEATURE_DRIFT_VISUALS)
if "'monitoring_run_id'" in feature_str:
    issues.append("❌ FEATURE_DRIFT_VISUALS still uses 'monitoring_run_id' (Cell 22 NOT re-run)")
elif "'drifted_columns_count'" in feature_str:
    print("✅ FEATURE_DRIFT_VISUALS: Uses drifted_columns_count (Cell 22 was re-run)")

print("\n" + "=" * 80)

if issues:
    print("\n⚠️  VARIABLES NOT UPDATED - Analysis will fail!")
    print("\nProblems found:")
    for issue in issues:
        print(f"  {issue}")
    
    print("\n🔧 TO FIX:")
    print("  1. Go back and run Cell 18 (defines INFERENCE_VISUALS)")
    print("  2. Go back and run Cell 20 (defines DRIFT_VISUALS)")
    print("  3. Go back and run Cell 22 (defines FEATURE_DRIFT_VISUALS)")
    print("  4. Re-run this diagnostic cell - should show all ✅")
    print("  5. Then run Cell 25 (analysis update)")
    
    print("\n⛔ DO NOT run Cell 25 until all checks show ✅")
else:
    print("\n✅ ALL VARIABLES UPDATED CORRECTLY")
    print("\n✓ Safe to run Cell 25 (analysis update)")
    print("  The analysis should complete successfully now.")

print("=" * 80)


CHECKING VARIABLE VALUES IN KERNEL MEMORY


✅ ALL VARIABLES UPDATED CORRECTLY

✓ Safe to run Cell 25 (analysis update)
  The analysis should complete successfully now.


In [20]:
ANALYSIS_ACTIONS = [
    'quicksight:RestoreAnalysis', 'quicksight:UpdateAnalysisPermissions',
    'quicksight:DeleteAnalysis', 'quicksight:DescribeAnalysisPermissions',
    'quicksight:QueryAnalysis', 'quicksight:DescribeAnalysis', 'quicksight:UpdateAnalysis',
]

analysis_definition = {
    'DataSetIdentifierDeclarations': [
        {'Identifier': 'inference-ds', 'DataSetArn': DATASET_ARN},
        {'Identifier': 'drift-ds', 'DataSetArn': DRIFT_DATASET_ARN},
        {'Identifier': 'feature-drift-ds', 'DataSetArn': FEATURE_DRIFT_DATASET_ARN},
        {'Identifier': 'feature-level-ds', 'DataSetArn': FEATURE_LEVEL_DATASET_ARN},
        {'Identifier': 'accuracy-ds', 'DataSetArn': ACCURACY_DATASET_ARN},
    ],
    'Sheets': [
        {
            'SheetId': 'governance-sheet-1',
            'Name': 'Inference Monitoring',
            'Visuals': INFERENCE_VISUALS,
        },
        {
            'SheetId': 'governance-sheet-2',
            'Name': 'Drift Trend Analysis',
            'Visuals': DRIFT_VISUALS,
        },
        {
            'SheetId': 'governance-sheet-3',
            'Name': 'Feature Drift Analysis',
            'Visuals': FEATURE_DRIFT_VISUALS,
        },
        {
            'SheetId': 'governance-sheet-4',
            'Name': 'Feature Drift Detail',
            'Visuals': FEATURE_LEVEL_VISUALS,
        },
    ],
    'AnalysisDefaults': {
        'DefaultNewSheetConfiguration': {
            'InteractiveLayoutConfiguration': {
                'FreeForm': {'CanvasSizeOptions': {'ScreenCanvasSizeOptions': {'OptimizedViewPortWidth': '1600px'}}}
            }
        }
    },
}

import time

try:
    quicksight.describe_analysis(AwsAccountId=ACCOUNT_ID, AnalysisId=ANALYSIS_ID)
    print('Analysis exists, updating...')
    resp = quicksight.update_analysis(
        AwsAccountId=ACCOUNT_ID, AnalysisId=ANALYSIS_ID,
        Name=QUICKSIGHT_ANALYSIS_NAME,
        Definition=analysis_definition,
    )
    
    # Wait for analysis to reach terminal state
    print('  Waiting for analysis to complete...')
    max_attempts = 30
    for attempt in range(max_attempts):
        analysis_resp = quicksight.describe_analysis(AwsAccountId=ACCOUNT_ID, AnalysisId=ANALYSIS_ID)
        status = analysis_resp['Analysis']['Status']
        
        if status == 'CREATION_SUCCESSFUL':
            print(f'  ✓ Analysis update successful')
            break
        elif status in ['CREATION_FAILED', 'UPDATE_FAILED', 'DELETED']:
            print(f'  ✗ Analysis update failed with status: {status}')
            
            # Get error details
            if 'Errors' in analysis_resp['Analysis']:
                errors = analysis_resp['Analysis']['Errors']
                print(f'\n  Errors ({len(errors)} total):')
                for i, err in enumerate(errors[:5], 1):  # Show first 5 errors
                    err_type = err.get('Type', 'Unknown')
                    err_msg = err.get('Message', 'No message')
                    print(f'    {i}. [{err_type}] {err_msg}')
                
                if len(errors) > 5:
                    print(f'    ... and {len(errors) - 5} more errors')
            
            raise Exception(f"Analysis in {status} state - see errors above")
        
        time.sleep(2)
    else:
        print(f'  ⚠ Timeout waiting for analysis (status: {status})')
        
except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceNotFoundException':
        print('Creating new analysis...')
        resp = quicksight.create_analysis(
            AwsAccountId=ACCOUNT_ID, AnalysisId=ANALYSIS_ID,
            Name=QUICKSIGHT_ANALYSIS_NAME,
            Definition=analysis_definition,
            Permissions=[{'Principal': p, 'Actions': ANALYSIS_ACTIONS} for p in QS_PRINCIPALS],
        )
        print('  ✓ Analysis created')
    else:
        raise

ANALYSIS_ARN = resp['Arn']
print(f'\n✓ Analysis: {ANALYSIS_ARN}')
print(f'  Open: https://{REGION}.quicksight.aws.amazon.com/sn/analyses/{ANALYSIS_ID}')

Analysis exists, updating...


  Waiting for analysis to complete...


  ⚠ Timeout waiting for analysis (status: UPDATE_SUCCESSFUL)

✓ Analysis: arn:aws:quicksight:us-east-1:329430715989:analysis/fraud-governance-analysis
  Open: https://us-east-1.quicksight.aws.amazon.com/sn/analyses/fraud-governance-analysis


## 9. Publish Dashboard via Definition API

In [ ]:
CONFIRM_DELETE = False  # set True to delete all governance QuickSight resources

if CONFIRM_DELETE:
    outcome = gov.delete_dashboard(region=AWS_DEFAULT_REGION)
    print('Deleted:  ', outcome['deleted'])
    print('Not found:', outcome['not_found'])
    print('Errors:   ', outcome['errors'])
else:
    print('Cleanup skipped — set CONFIRM_DELETE = True to run.')

Cleanup cell — uncomment and set CONFIRM_DELETE = True to run
